In [1]:
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
# Grog model

model = init_chat_model("groq:qwen/qwen3-32b")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.messages import HumanMessage

In [ ]:
def read_email_tool(email_id:str)->str:
    """ Mock function to read email by id"""
    return f"Email content for Id: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """ Mock function to send email by id"""
    return f"Email sent to {recipient} with subject {subject}"




In [10]:
agent = create_agent(
    model ="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer= InMemorySaver(),
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject", "edit"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [11]:
config={"configurable":{"thread_id":"test-approve"}}

In [12]:
#step1: request
result =agent.invoke({
    "messages":[HumanMessage(content="send email to john@email.com with subject 'Hello and body 'How are you'")]
},
config = config)
result

{'messages': [HumanMessage(content="send email to john@email.com with subject 'Hello and body 'How are you'", additional_kwargs={}, response_metadata={}, id='43be1852-1f68-4247-9133-c819cb020e5c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to john@email.com with the subject 'Hello' and body 'How are you'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the user is asking to send an email, the send_email_tool is the right choice.\n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is john@email.com, subject is 'Hello', and body is 'How are you'. I should make sure all required parameters are included. Yep, they're all there. So I need to structure the tool call with these arguments in JSON format. Let me double-check the parameter names and types. The recipient is a string, subjec

In [13]:
#step2: approve

if "__interrupt__" in result:
    print("Paused! Approving..")

    result = agent.invoke(
        Command(resume={"decisions":[{"type":"approve"}]}),
        config=config
    )
    print(f"Reult: {result['messages'][-1].content}")
    

Paused! Approving..
Reult: ✅ The email with the subject "Hello" has been successfully sent to john@email.com. Let me know if you need anything else!


In [14]:
result

{'messages': [HumanMessage(content="send email to john@email.com with subject 'Hello and body 'How are you'", additional_kwargs={}, response_metadata={}, id='43be1852-1f68-4247-9133-c819cb020e5c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to john@email.com with the subject 'Hello' and body 'How are you'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the user is asking to send an email, the send_email_tool is the right choice.\n\nLooking at the parameters for send_email_tool, it requires recipient, subject, and body. The user provided all three: recipient is john@email.com, subject is 'Hello', and body is 'How are you'. I should make sure all required parameters are included. Yep, they're all there. So I need to structure the tool call with these arguments in JSON format. Let me double-check the parameter names and types. The recipient is a string, subjec

In [ ]:
##reject

def read_email_tool(email_id:str)->str:
    """ Mock function to read email by id"""
    return f"Email content for Id: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """ Mock function to send email by id"""
    return f"Email sent to {recipient} with subject {subject}"

agent = create_agent(
    model ="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer= InMemorySaver(),
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject", "edit"]
                },
                "read_email_tool":False
            }
        )
    ]
)


In [16]:
config={"configurable":{"thread_id":"test-reject"}}

In [17]:
#step1: request
result =agent.invoke({
    "messages":[HumanMessage(content="send email to john@email.com with subject 'Hello and body 'How are you'")]
},
config = config)
result

{'messages': [HumanMessage(content="send email to john@email.com with subject 'Hello and body 'How are you'", additional_kwargs={}, response_metadata={}, id='d1486d28-fff5-43ea-b6a1-df646459ab26'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@email.com with the subject 'Hello' and body 'How are you'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. The parameters are all required, so I need to make sure each one is included. The recipient is john@email.com, subject is Hello, and body is How are you. I'll structure the function call with these details. No need for the read_email_tool here since the user isn't asking to read an email. Just need to call send_email_tool with the provided parameters.\n", 'tool_calls': [{'id': 't1j5eee12', 'function': {'arguments': '{"body":"How are you","recipient":"john@email.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type

In [18]:
#step2: Reject

if "__interrupt__" in result:
    print("Paused! Approving..")

    result = agent.invoke(
        Command(resume={"decisions":[{"type":"reject"}]}),
        config=config
    )
    print(f"Reult: {result['messages'][-1].content}")
    

Paused! Approving..
Reult: The request to send an email to `john@email.com` with the subject "Hello" and body "How are you" was rejected (ID: **t1j5eee12**). This could be due to a system error or invalid input. Would you like me to:

1. Attempt the request again?
2. Check if the email address or content needs correction?
3. Provide further details about this rejection?

Let me know how you'd like to proceed.


In [19]:
result

{'messages': [HumanMessage(content="send email to john@email.com with subject 'Hello and body 'How are you'", additional_kwargs={}, response_metadata={}, id='d1486d28-fff5-43ea-b6a1-df646459ab26'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@email.com with the subject 'Hello' and body 'How are you'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. The parameters are all required, so I need to make sure each one is included. The recipient is john@email.com, subject is Hello, and body is How are you. I'll structure the function call with these details. No need for the read_email_tool here since the user isn't asking to read an email. Just need to call send_email_tool with the provided parameters.\n", 'tool_calls': [{'id': 't1j5eee12', 'function': {'arguments': '{"body":"How are you","recipient":"john@email.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type

In [20]:
##Editing
def read_email_tool(email_id:str)->str:
    """ Mock function to read email by id"""
    return f"Email content for Id: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """ Mock function to send email by id"""
    return f"Email sent to {recipient} with subject {subject}"

agent = create_agent(
    model ="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer= InMemorySaver(),
    middleware =[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject", "edit"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [21]:
config={"configurable":{"thread_id":"test-edit"}}

In [22]:
#step1: request
result =agent.invoke({
    "messages":[HumanMessage(content="send email to wrong@email.com with subject 'test' and body 'Hello'")]
},
config = config)
result

{'messages': [HumanMessage(content="send email to wrong@email.com with subject 'test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='9cda3bb4-6c1b-41d5-97dc-c18d6f3cb506'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required, and they're all present. So I should call the send_email_tool with those arguments. No issues here, even though the email might be invalid, but that's up to the tool to handle. Just need to pass the parameters correctly.\n", 'tool_calls': [{'id': 'j6ft9kwhs', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"t

In [23]:
#step2: edit and approve

if "__interrupt__" in result:
    print("Paused! Editing..")

    result = agent.invoke(
        Command(resume={"decisions":[
            {"type":"edit",
             "edited_action":{
                 "name":"send_email_tool",
                 "args":{
                     "recipient":"correct@email.com",
                     "subject":"Corrected subject",
                     "body":"This was edited by human before sending"
                 }
             }}
        ]}),
        config=config
    )
    print(f"Reult: {result['messages'][-1].content}")
    

Paused! Editing..
Reult: The email has been successfully sent to **correct@email.com** with the subject **"Corrected subject"** and an edited body. Let me know if you need further assistance!


In [24]:
result

{'messages': [HumanMessage(content="send email to wrong@email.com with subject 'test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='9cda3bb4-6c1b-41d5-97dc-c18d6f3cb506'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required, and they're all present. So I should call the send_email_tool with those arguments. No issues here, even though the email might be invalid, but that's up to the tool to handle. Just need to pass the parameters correctly.\n", 'tool_calls': [{'id': 'j6ft9kwhs', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"t